In [33]:
import numpy as np
import re
import chromadb

### Loading a Pretrained Sentence Transformer Model

To generate text embeddings, we use the pretrained **all-MiniLM-L6-v2** model from the Sentence Transformers library.

```python
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
```

#### Why this model?

* It is lightweight and efficient, making it suitable for experimentation and production use.
* It converts sentences into dense vector representations (**embeddings**) with a dimension of **384**.
* Sentences with similar meanings are mapped to nearby locations in the embedding space.
* It is commonly used for tasks such as:

  * Semantic search
  * Text similarity
  * Clustering
  * Information retrieval
  * Recommendation systems

After loading the model, each sentence can be transformed into a 384-dimensional embedding vector that captures its semantic meaning.


In [34]:
from sentence_transformers import SentenceTransformer

# 1. Load a pretrained Sentence Transformer model
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7579.05it/s]


In [35]:
sentences = [
    "I am scared about the rest of my professional path.",
    "My career path is not clear at the moment, but with effort and persistence, I can move in the right direction.",
    "I like to know more about Germany.",
]

# 2. Calculate embeddings by calling model.encode()
embeddings = model.encode(sentences)
embeddings.shape

(3, 384)

# Use Buit-in similarity Function

In [36]:
# 3. Calculate the embedding similarities
similarities = model.similarity(embeddings, embeddings)
print(similarities)

tensor([[1.0000, 0.5275, 0.0500],
        [0.5275, 1.0000, 0.0487],
        [0.0500, 0.0487, 1.0000]])


### Using NumPy Instead of the Built-in `similarity` Function

To check the similarity between two embeddings, we can implement **Cosine Similarity** ourselves using NumPy instead of relying on an internal similarity function.

Cosine Similarity measures how close two vectors are by calculating the cosine of the angle between them.

$$
\text{Cosine Similarity}(A,B)
=
\frac{A \cdot B}
{\|A\|\,\|B\|}
$$

The expanded form is:

$$
\cos(\theta)
=
\frac{\sum_{i=1}^{n} A_iB_i}
{\sqrt{\sum_{i=1}^{n} A_i^2}
 \sqrt{\sum_{i=1}^{n} B_i^2}}
$$

#### Key Points

* **Direction over magnitude:** Cosine Similarity focuses on the direction of the vectors rather than their magnitude or length.
* **Value range:** The result ranges from **-1 to 1**.
* **Interpretation:**

  * **1** → The vectors point in exactly the same direction → highly similar.
  * **0** → The vectors are perpendicular → no directional similarity.
  * **-1** → The vectors point in completely opposite directions.

This makes Cosine Similarity particularly useful for comparing **embeddings**, where the direction of the vector represents semantic information.


In [37]:
def cosine_sim(vec1, vec2):

    dot_product = np.dot(vec1 , vec2)
    norm = np.linalg.norm(vec1) * np.linalg.norm(vec2)
    cosine_similarity = dot_product / norm
    return cosine_similarity


In [38]:
print(cosine_sim(embeddings[0], embeddings[1]))
print(cosine_sim(embeddings[0], embeddings[2]))
print(cosine_sim(embeddings[1], embeddings[2]))

# %%


0.5274734
0.050040323
0.048657864


In [39]:
ranking = {}

query = "Every body should make your own decision"
embeddings_query = model.encode(query)

for i, sentence in enumerate(sentences):
    score = cosine_sim(embeddings_query , embeddings[i])
    ranking[sentence] = score

ranking_sorted = sorted(
    ranking.items(),
    key=lambda x: x[1],
    reverse=True
)

In [40]:
print(ranking_sorted)

[('My career path is not clear at the moment, but with effort and persistence, I can move in the right direction.', np.float32(0.2622649)), ('I am scared about the rest of my professional path.', np.float32(0.212736)), ('I like to know more about Germany.', np.float32(0.053430203))]


In [41]:
for sentence, score in ranking_sorted:
    print(f"{score:.4f} | {sentence}")

0.2623 | My career path is not clear at the moment, but with effort and persistence, I can move in the right direction.
0.2127 | I am scared about the rest of my professional path.
0.0534 | I like to know more about Germany.


## Text Loading and Initial Chunking

In this step, the text of **Article 5 of the EU AI Act** was loaded from a local text file and inspected before applying any chunking strategy.

### 1. Load the text file

The article was loaded using Python's `open()` function with a context manager:

```python
with open('data/ai_act_article5.txt', 'r') as file:
    text = file.read()
```

Using a relative path makes the notebook more portable and allows it to work when the project is cloned to another machine, assuming the project structure remains the same.

### 2. Inspect the extracted text

The type and beginning of the text were checked:

```python
print(type(text))
print(text[:1000])
```

The text was confirmed to be stored as a Python `str`.

The total number of characters was also checked:

```python
print(len(text))
```

To inspect invisible characters such as newlines and spaces, `repr()` was used:

```python
print(repr(text[:300]))
```

The inspection showed that the document contains actual newline characters and that major sections are separated by `\n\n`.

### 3. Inspect newline structure

The number of newline characters was checked:

```python
print(text.count('\n'))
```

This confirmed that the text contains paragraph-style line breaks.

However, a simple split using:

```python
text.split('\n\n')
```

would not produce ideal semantic chunks because the lettered sections of Article 5 are structured like:

```text
(a)

the placing on the market...

(b)

...
```

A simple `\n\n` split would therefore separate `(a)` from its corresponding content.

### 4. Split using lettered sections

Because Article 5 contains clearly defined lettered points such as `(a)`, `(b)`, `(c)`, these markers were used as natural boundaries for the next stage of chunking.

Python's `re` module was used with `re.split()`:

```python
import re

parts = re.split(r'\n\n(\([a-z]\))\n\n', text)

print(len(parts))
```

The regular expression identifies markers such as:

```text
(a)
(b)
(c)
```

while the capturing group:

```text
(\([a-z]\))
```

keeps the lettered marker in the resulting list.

The resulting structure is therefore expected to alternate between:

```text
intro text
(a)
content of (a)
(b)
content of (b)
...
```

This intermediate representation will be recombined in the next step so that each lettered marker stays attached to its corresponding content.

### Current status

At this point:

* The Article 5 text has been loaded successfully.
* A relative file path is being used.
* The text structure has been inspected.
* Newline characters and formatting have been examined.
* A simple `\n\n` split was evaluated and found unsuitable for semantic chunking.
* A regex-based split was created to identify the lettered sections.
* The lettered markers are preserved using a capturing group.

**Next step:** recombine the alternating elements of `parts` into coherent chunks such as `(a) + content`, `(b) + content`, and so on.


In [42]:
with open('data/ai_act_article5.txt' , 'r') as file: 
    text = file.read()

In [43]:
print(f'Type of our text file is :{type(text)}')
print('--' * 30)
print(f'Length of our text file is :{len(text)}')
print('--' * 30)
print(text[:1000])


Type of our text file is :<class 'str'>
------------------------------------------------------------
Length of our text file is :11172
------------------------------------------------------------
Article 5

Prohibited AI practices

1.   The following AI practices shall be prohibited:

(a)

the placing on the market, the putting into service or the use of an AI system that deploys subliminal techniques beyond a person’s consciousness or purposefully manipulative or deceptive techniques, with the objective, or the effect of materially distorting the behaviour of a person or a group of persons by appreciably impairing their ability to make an informed decision, thereby causing them to take a decision that they would not have otherwise taken in a manner that causes or is reasonably likely to cause that person, another person or group of persons significant harm;

(b)

the placing on the market, the putting into service or the use of an AI system that exploits any of the vulnerabilities of 

In [44]:
text.count('\n')

84

In [45]:
repr(text[:300])

"'Article 5\\n\\nProhibited AI practices\\n\\n1.   The following AI practices shall be prohibited:\\n\\n(a)\\n\\nthe placing on the market, the putting into service or the use of an AI system that deploys subliminal techniques beyond a person’s consciousness or purposefully manipulative or deceptive techniques, with '"

In [46]:
parts = re.split(r'\n\n(\([a-z]\))\n\n', text) 
print(len(parts))

25


In [47]:
print(parts[:5])

['Article 5\n\nProhibited AI practices\n\n1.   The following AI practices shall be prohibited:', '(a)', 'the placing on the market, the putting into service or the use of an AI system that deploys subliminal techniques beyond a person’s consciousness or purposefully manipulative or deceptive techniques, with the objective, or the effect of materially distorting the behaviour of a person or a group of persons by appreciably impairing their ability to make an informed decision, thereby causing them to take a decision that they would not have otherwise taken in a manner that causes or is reasonably likely to cause that person, another person or group of persons significant harm;', '(b)', 'the placing on the market, the putting into service or the use of an AI system that exploits any of the vulnerabilities of a natural person or a specific group of persons due to their age, disability or a specific social or economic situation, with the objective, or the effect, of materially distorting t

## 4. Recombining Labels and Contents into Final Chunks

After splitting the text with regular expressions, the labels `(a)`, `(b)`, etc. and their corresponding content were stored separately.

The labels were extracted using:

```python
labels = parts[1::2]
```

and the contents using:

```python
contents = parts[2::2]
```

Each label was then combined with its corresponding content using `zip()`:

```python
chunks = [parts[0]]

for label, content in zip(labels, contents):
    chunks.append(' '.join([label, content]))
```

Here, `zip(labels, contents)` pairs each label with the correct content:

```text
(a) → text of (a)
(b) → text of (b)
(c) → text of (c)
...
```

The `join()` method combines the label and content into a single string with a space between them.

For example:

```text
(a) the placing on the market, the putting into service or the use of an AI system...
```

Finally, the result was checked:

```python
print(len(chunks))
print(chunks[1])
```

The expected number of chunks is **13**:

* `chunks[0]` → Article 5 introduction
* `chunks[1]` → `(a)` + its content
* `chunks[2]` → `(b)` + its content
* ...
* `chunks[12]` → `(l)` + its content

At this point, the original Article 5 text has been transformed into meaningful text chunks that can be used in the next stage of the project: **generating embeddings and comparing chunks using cosine similarity**.


In [48]:
chunks = [parts[0]]
labels = parts[1::2]
contents = parts[2::2]

print(labels)
print(contents[:3])

['(a)', '(b)', '(c)', '(i)', '(d)', '(e)', '(f)', '(g)', '(h)', '(i)', '(a)', '(b)']
['the placing on the market, the putting into service or the use of an AI system that deploys subliminal techniques beyond a person’s consciousness or purposefully manipulative or deceptive techniques, with the objective, or the effect of materially distorting the behaviour of a person or a group of persons by appreciably impairing their ability to make an informed decision, thereby causing them to take a decision that they would not have otherwise taken in a manner that causes or is reasonably likely to cause that person, another person or group of persons significant harm;', 'the placing on the market, the putting into service or the use of an AI system that exploits any of the vulnerabilities of a natural person or a specific group of persons due to their age, disability or a specific social or economic situation, with the objective, or the effect, of materially distorting the behaviour of that pers

In [49]:
for label, content in zip(labels, contents):
    chunks.append(' '.join([label , content]))

In [50]:
print(len(chunks))
print(chunks[1])

13
(a) the placing on the market, the putting into service or the use of an AI system that deploys subliminal techniques beyond a person’s consciousness or purposefully manipulative or deceptive techniques, with the objective, or the effect of materially distorting the behaviour of a person or a group of persons by appreciably impairing their ability to make an informed decision, thereby causing them to take a decision that they would not have otherwise taken in a manner that causes or is reasonably likely to cause that person, another person or group of persons significant harm;


## Vector Storage & Retrieval

## Vector Database Setup

Create a persistent Chroma client to store embeddings and documents locally.

The database is stored in the `vector_store/` directory so that the collection persists between notebook sessions.

In [51]:
client = chromadb.PersistentClient(path='vector_store')

## Create a Chroma Collection

Create a collection for the EU AI Act Article 5 chunks.

A collection is a named space in Chroma where related documents and their embeddings are stored.

In [ ]:
collection = client.create_collection(name='ai_act_article5')

## Generate Chunk Embeddings

Generate an embedding vector for each Article 5 chunk using the pretrained `all-MiniLM-L6-v2` model.

Each chunk is represented as a 384-dimensional vector, allowing us to compare the semantic similarity between chunks and queries.

In [53]:
chunks_embedding = model.encode(chunks)

print(chunks_embedding.shape)
print(len(chunks))


(13, 384)
13


## Store Chunks and Embeddings

Store the Article 5 chunks and their corresponding embeddings in the Chroma collection.

Each chunk receives a unique ID so that it can be identified when retrieved later.

## Verify Collection

Check the number of documents stored in the Chroma collection.

The collection should contain 13 chunks: one introductory chunk and 12 chunks corresponding to the prohibited practices `(a)` through `(l)`.

In [54]:
collection.add(
    ids = [f'chunk_{i}' for i in range(len(chunks))],
    documents = chunks,
    embeddings = chunks_embedding
)

print(collection.count())

13


## Test Retrieval with a Query

Test semantic retrieval using a natural-language question:

> "What counts as manipulating someone's behavior?"

The query is converted into an embedding and compared against the stored Article 5 chunk embeddings.

## Generate Query Embedding

Convert the user's query into the same embedding space as the stored document chunks.

Using the same embedding model allows Chroma to compare the query vector with the chunk vectors.

## Retrieve Relevant Chunks

Query the Chroma collection and retrieve the three chunks with the smallest distances to the query embedding.

Lower distance indicates greater similarity between the query and the retrieved chunk.

In [64]:
query = "What counts as manipulating someone's behavior?"

query_embedding = model.encode(query)

In [65]:
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3
)

## Inspect Retrieval Results

Inspect the retrieved chunk IDs, documents, and distances to understand which parts of Article 5 were considered most relevant to the query.

This allows us to evaluate whether the retrieval result matches our expectations.

In [66]:
print(results)

{'ids': [['chunk_4', 'chunk_1', 'chunk_2']], 'embeddings': None, 'documents': [['(i) detrimental or unfavourable treatment of certain natural persons or groups of persons in social contexts that are unrelated to the contexts in which the data was originally generated or collected;\n\n(ii)\n\ndetrimental or unfavourable treatment of certain natural persons or groups of persons that is unjustified or disproportionate to their social behaviour or its gravity;', '(a) the placing on the market, the putting into service or the use of an AI system that deploys subliminal techniques beyond a person’s consciousness or purposefully manipulative or deceptive techniques, with the objective, or the effect of materially distorting the behaviour of a person or a group of persons by appreciably impairing their ability to make an informed decision, thereby causing them to take a decision that they would not have otherwise taken in a manner that causes or is reasonably likely to cause that person, anoth

## Compare Distance Metrics

Create a second Chroma collection using cosine distance instead of the default L2 (Euclidean) distance.

This allows us to investigate whether the choice of distance metric affects the retrieval ranking.

## Store Embeddings in the Cosine Collection

Store the same Article 5 chunks and embeddings in the cosine-distance collection.

Keeping the documents and embeddings identical allows us to compare L2 and cosine retrieval fairly.

## Store Embeddings in the Cosine Collection

Store the same Article 5 chunks and embeddings in the cosine-distance collection.

Keeping the documents and embeddings identical allows us to compare L2 and cosine retrieval fairly.

## Test Retrieval with Cosine Distance

Run the same query against the cosine-distance collection and compare its ranking and distances with the L2 collection.

The goal is to determine whether changing the distance metric changes the retrieved results.

## L2 vs. Cosine Retrieval

Compare the retrieval rankings produced by the L2 and cosine collections.

For this experiment, both metrics returned the same ranking:

1. `chunk_4`
2. `chunk_1`
3. `chunk_2`

Although the distance values are different, the ranking is unchanged.

This suggests that the distance metric is not responsible for the unexpected top result in this case.

In [ ]:
cosine_collection = client.create_collection(

                name='ai_act_article5_cosine',
                metadata={"hnsw:space": "cosine"}
    
    )

In [ ]:
cosine_collection.add(

    ids=[f'chunk_{i}' for i in range(len(chunks))],
    documents=chunks,
    embeddings=chunks_embedding
)

print(cosine_collection.count())

13


In [ ]:
cosine_results = cosine_collection.query(
    query_embeddings=[query_embedding],
    n_results=3
)

print(cosine_results)

{'ids': [['chunk_4', 'chunk_1', 'chunk_2']], 'embeddings': None, 'documents': [['(i) detrimental or unfavourable treatment of certain natural persons or groups of persons in social contexts that are unrelated to the contexts in which the data was originally generated or collected;\n\n(ii)\n\ndetrimental or unfavourable treatment of certain natural persons or groups of persons that is unjustified or disproportionate to their social behaviour or its gravity;', '(a) the placing on the market, the putting into service or the use of an AI system that deploys subliminal techniques beyond a person’s consciousness or purposefully manipulative or deceptive techniques, with the objective, or the effect of materially distorting the behaviour of a person or a group of persons by appreciably impairing their ability to make an informed decision, thereby causing them to take a decision that they would not have otherwise taken in a manner that causes or is reasonably likely to cause that person, anoth

## Query Phrasing Experiment

The first query produced an unexpected top result. To investigate whether retrieval is sensitive to query wording, test the same collection with a more precise query.

In [ ]:
query_2 = "deceptive AI techniques that distort behavior"
query_2_embedding = model.encode(query_2)

In [ ]:
results_2_l2 = collection.query(
    query_embeddings=[query_2_embedding],
    n_results=3
)

print(results_2_l2["ids"])
print(results_2_l2["distances"])

[['chunk_1', 'chunk_5', 'chunk_0']]
[[1.0822088718414307, 1.3176766633987427, 1.3378362655639648]]


In [ ]:
results_2_cosine = cosine_collection.query(
    query_embeddings=[query_2_embedding],
    n_results=3
)

print(results_2_cosine["ids"])
print(results_2_cosine["distances"])

[['chunk_1', 'chunk_5', 'chunk_0']]
[[0.5411045551300049, 0.6588382720947266, 0.6689181327819824]]


## Conclusion

Built a complete semantic retrieval pipeline using embeddings and ChromaDB to search Article 5 of the EU AI Act.

I tested both L2 and cosine distance and found that they produced the same retrieval ranking. The experiments also showed that retrieval quality depends strongly on query phrasing: a more precise query was able to retrieve the most relevant chunk correctly.

In [71]:
def retrieve(query , n = 3):

    query_embedding = model.encode(query)
    results = cosine_collection.query(
            query_embeddings= [query_embedding],
            n_results=n
         )
    documents = results['documents'][0]
    distances = results['distances'][0]

    return(list(zip(documents,distances)))


In [72]:
retrieve("deceptive AI techniques that distort behavior")

[('(a) the placing on the market, the putting into service or the use of an AI system that deploys subliminal techniques beyond a person’s consciousness or purposefully manipulative or deceptive techniques, with the objective, or the effect of materially distorting the behaviour of a person or a group of persons by appreciably impairing their ability to make an informed decision, thereby causing them to take a decision that they would not have otherwise taken in a manner that causes or is reasonably likely to cause that person, another person or group of persons significant harm;',
  0.5411045551300049),
 ('(d) the placing on the market, the putting into service for this specific purpose, or the use of an AI system for making risk assessments of natural persons in order to assess or predict the risk of a natural person committing a criminal offence, based solely on the profiling of a natural person or on assessing their personality traits and characteristics; this prohibition shall not